# BBB-Relevant PaDEL Descriptor Training
GridSearchCV tuning with cross-validation, restricted to descriptors most relevant to BBB permeability.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler

try:
    from imblearn.pipeline import Pipeline
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    IMBLEARN_AVAILABLE = True
except Exception as exc:
    Pipeline = SkPipeline
    IMBLEARN_AVAILABLE = False
    print(f'imblearn not available; sampler search will use class-weighted/no-sampler models only: {exc}')

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGB_AVAILABLE = False
    print(f'xgboost not available; skipping XGB: {exc}')

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception as exc:
    LGBM_AVAILABLE = False
    print(f'lightgbm not available; skipping LGBM: {exc}')

RANDOM_SEEDS = [42, 101, 202]
BASE_SEED = RANDOM_SEEDS[0]
CV_FOLDS = 3
N_JOBS = 1

OUTPUT_DIR = Path('../output/models/bbb_relevant_padel_v2')
FIGURE_DIR = Path('../figures/new_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_split(split_ratio='8020', base_dir='../data/split'):
    base = Path(base_dir) / split_ratio
    X_train = pd.read_csv(base / 'x_train.csv', index_col=0)
    y_train = pd.read_csv(base / 'y_train.csv', index_col=0).iloc[:, 0].astype(int)
    X_test = pd.read_csv(base / 'x_test.csv', index_col=0)
    y_test = pd.read_csv(base / 'y_test.csv', index_col=0).iloc[:, 0].astype(int)
    return X_train, y_train, X_test, y_test


def keep_numeric_features(X_train, X_test):
    X_train_num = X_train.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    X_test_num = X_test[X_train_num.columns].replace([np.inf, -np.inf], np.nan)
    return X_train_num, X_test_num


def valid_k_values(n_features):
    candidates = [250, 'all']
    return [k for k in candidates if k == 'all' or k < n_features]


def make_sampler(seed, kind):
    if kind == 'none' or not IMBLEARN_AVAILABLE:
        return 'passthrough'
    samplers = {
        'under': RandomUnderSampler(random_state=seed, sampling_strategy=0.85),
        'tomek': TomekLinks(),
        'smote': SMOTE(random_state=seed, sampling_strategy=0.85, k_neighbors=5),
        'borderline_smote': BorderlineSMOTE(random_state=seed, sampling_strategy=0.85, k_neighbors=5),
    }
    return samplers[kind]


def make_model(model_name, seed, y=None):
    if model_name == 'LogReg':
        return LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed, solver='liblinear')
    if model_name == 'KNN':
        return KNeighborsClassifier()
    if model_name == 'RF':
        return RandomForestClassifier(random_state=seed, class_weight='balanced_subsample', n_jobs=1)
    if model_name == 'ET':
        return ExtraTreesClassifier(random_state=seed, class_weight='balanced', n_jobs=1)
    if model_name == 'HGB':
        return HistGradientBoostingClassifier(random_state=seed, l2_regularization=0.1)
    if model_name == 'XGB' and XGB_AVAILABLE:
        neg = int((y == 0).sum()) if y is not None else 1
        pos = int((y == 1).sum()) if y is not None else 1
        return XGBClassifier(
            random_state=seed, eval_metric='logloss', tree_method='hist', n_jobs=1,
            scale_pos_weight=max(1.0, neg / max(pos, 1)), verbosity=0
        )
    if model_name == 'LGBM' and LGBM_AVAILABLE:
        return LGBMClassifier(random_state=seed, class_weight='balanced', n_jobs=1, verbose=-1)
    raise ValueError(f'Unsupported or unavailable model: {model_name}')


def make_pipeline(model_name, seed, n_features, y=None):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('variance', VarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler()),
        ('sampler', 'passthrough'),
        ('selector', SelectKBest(mutual_info_classif, k=min(250, n_features))),
        ('clf', make_model(model_name, seed, y=y)),
    ])


def param_grid(model_name, n_features):
    if not IMBLEARN_AVAILABLE:
        sampler_choices = ['none']
    elif model_name in ['LogReg', 'KNN']:
        sampler_choices = ['none', 'under']
    else:
        sampler_choices = ['none', 'under', 'smote', 'borderline_smote']
    sampler_objects = [make_sampler(BASE_SEED, s) for s in sampler_choices]
    base = {
        'sampler': sampler_objects,
        'selector__k': valid_k_values(n_features),
    }
    if model_name == 'LogReg':
        base.update({'clf__C': [0.05, 0.1, 1, 10]})
    elif model_name == 'KNN':
        base.update({'clf__n_neighbors': [7, 15, 23], 'clf__weights': ['distance'], 'clf__p': [1, 2]})
    elif model_name == 'RF':
        base.update({'clf__n_estimators': [300, 500], 'clf__max_depth': [None, 24], 'clf__min_samples_leaf': [1, 3], 'clf__max_features': ['sqrt']})
    elif model_name == 'ET':
        base.update({'clf__n_estimators': [400, 600], 'clf__max_depth': [None, 32], 'clf__min_samples_leaf': [1, 2], 'clf__max_features': ['sqrt']})
    elif model_name == 'HGB':
        base.update({'clf__learning_rate': [0.05, 0.1], 'clf__max_iter': [350, 550], 'clf__max_leaf_nodes': [31, 63], 'clf__min_samples_leaf': [20]})
    elif model_name == 'XGB':
        base.update({'clf__n_estimators': [500, 750], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.05, 0.1], 'clf__subsample': [0.85], 'clf__colsample_bytree': [0.8], 'clf__reg_lambda': [1]})
    elif model_name == 'LGBM':
        base.update({'clf__n_estimators': [500, 750], 'clf__num_leaves': [31, 63], 'clf__learning_rate': [0.05, 0.1], 'clf__subsample': [0.85], 'clf__colsample_bytree': [0.8], 'clf__min_child_samples': [20]})
    return base


def clean_param_value(value):
    if value == 'passthrough':
        return 'none'
    if hasattr(value, '__class__') and value.__class__.__module__.startswith('imblearn'):
        return value.__class__.__name__
    if isinstance(value, np.generic):
        return value.item()
    return value


def clean_best_params(params):
    return {key: clean_param_value(value) for key, value in params.items()}


def best_cv_metrics(search):
    best_idx = search.best_index_
    cv = search.cv_results_
    metrics = {}
    for metric in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        metrics[f'cv_{metric}_mean'] = cv[f'mean_test_{metric}'][best_idx]
        metrics[f'cv_{metric}_std'] = cv[f'std_test_{metric}'][best_idx]
    return metrics


def predict_scores(model, X):
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        raw = model.decision_function(X)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    else:
        y_score = y_pred.astype(float)
    return y_pred, y_score


def compute_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'average_precision': average_precision_score(y_true, y_score),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }


def tune_model(model_name, X_train, y_train):
    n_features = X_train.shape[1]
    pipe = make_pipeline(model_name, BASE_SEED, n_features, y=y_train)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=BASE_SEED)
    scoring = {
        'accuracy': 'accuracy',
        'balanced_accuracy': 'balanced_accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
    }
    search = GridSearchCV(
        pipe,
        param_grid=param_grid(model_name, n_features),
        scoring=scoring,
        cv=cv,
        n_jobs=N_JOBS,
        refit='roc_auc',
        return_train_score=True,
        verbose=1,
    )
    search.fit(X_train, y_train)
    return search


BBB_RELEVANT_PATTERNS = [
    'alogp', 'logp', 'xlogp', 'mlogp', 'tpsa', 'psa', 'hba', 'hbd', 'nrot', 'rotb',
    'mw', 'amw', 'nheavyatom', 'natom', 'apol', 'amr', 'na', 'ndon', 'nacc',
    'arom', 'ring', 'lipinski', 'ghose', 'molar', 'polar', 'charge', 'chi', 'kappa',
    'petitjean', 'zagreb', 'estate', 'vsa', 'slogp', 'smr', 'peoe', 'mpeoe', 'bcute',
    'nacid', 'nbasic', 'nhetero', 'noxygen', 'nnitrogen', 'halogen'
]
BBB_RELEVANT_EXACT = {
    'ALogP', 'ALogp2', 'AMR', 'apol', 'TopoPSA', 'MW', 'nAtom', 'nHeavyAtom', 'nH',
    'nN', 'nO', 'nS', 'nP', 'nF', 'nCl', 'nBr', 'nI', 'nX', 'nAcid', 'nBase',
    'nRotB', 'nHBAcc', 'nHBDon', 'naAromAtom', 'nAromBond', 'nRing', 'nSmallRings',
    'nAromRings', 'nHeteroRing', 'nAromHeteroRing', 'MLogP', 'XLogP', 'LipinskiFailures'
}


def load_bbb_relevant_padel(path='../data/padel_results_with_bbb.csv'):
    df = pd.read_csv(path)
    df = df.dropna(subset=['BBB']).copy()
    y = df['BBB'].astype(int)
    non_features = {'BBB', 'smiles', 'Original_Name', 'Original_SMILES', 'Name'}
    numeric = df.drop(columns=[c for c in non_features if c in df.columns], errors='ignore')
    numeric = numeric.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)

    selected = []
    for col in numeric.columns:
        low = col.lower()
        if col in BBB_RELEVANT_EXACT or any(pattern in low for pattern in BBB_RELEVANT_PATTERNS):
            selected.append(col)

    if len(selected) < 30:
        selected = numeric.var(numeric_only=True).sort_values(ascending=False).head(120).index.tolist()

    X = numeric[selected]
    return X, y, selected


## Load BBB-Relevant PaDEL Descriptor Matrix

In [4]:
X_all, y_all, descriptor_cols = load_bbb_relevant_padel('../data/padel_loop_results_BBB.csv')
print(f'Dataset: {X_all.shape[0]} molecules | BBB relevant descriptors: {X_all.shape[1]}')
print(y_all.value_counts().sort_index())
print(descriptor_cols[:40])

(OUTPUT_DIR / 'bbb_relevant_descriptor_names.json').write_text(json.dumps({'features': descriptor_cols}, indent=2))
joblib.dump(descriptor_cols, OUTPUT_DIR / 'bbb_relevant_descriptor_names.pkl')

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=BASE_SEED)
train_idx, test_idx = next(splitter.split(X_all, y_all))
X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
print(f'Train: {X_train.shape} | Test: {X_test.shape}')


Dataset: 9584 molecules | BBB relevant descriptors: 169
BBB
0    2792
1    6792
Name: count, dtype: int64
['nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nH', 'nN', 'nO', 'nS', 'nP', 'nF', 'nCl', 'nBr', 'nI', 'nX', 'nBase', 'CrippenLogP', 'nHBd', 'nwHBd', 'nHBa', 'nwHBa', 'naaCH', 'naasC', 'naaaC', 'naaNH', 'naaN', 'naasN', 'naaO', 'naOm', 'naaS', 'naaSe', 'SHBd', 'SwHBd', 'SHBa', 'SwHBa', 'minHBd']
Train: (7667, 169) | Test: (1917, 169)


## GridSearchCV Hyperparameter Tuning with Cross-Validation

In [5]:
# Skip LogReg and KNN (already trained)
model_names = ['RF', 'ET', 'HGB']
if XGB_AVAILABLE:
    model_names.append('XGB')
if LGBM_AVAILABLE:
    model_names.append('LGBM')

searches = {}
official_rows = []
hyperparameter_rows = []
prediction_frames = []

for model_name in model_names:
    print('\n' + '=' * 80)
    print(f'Tuning {model_name} with GridSearchCV + {CV_FOLDS}-fold Stratified CV')
    search = tune_model(model_name, X_train, y_train)
    searches[model_name] = search
    best_model = search.best_estimator_
    best_params_clean = clean_best_params(search.best_params_)
    cv_metrics = best_cv_metrics(search)

    y_pred, y_score = predict_scores(best_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    row = {
        'model': model_name,
        'split': 'bbb_relevant_8020_official',
        'best_cv_roc_auc': search.best_score_,
        'best_params': str(best_params_clean),
        **cv_metrics,
        **metrics,
    }
    official_rows.append(row)
    hyperparameter_rows.append({'model': model_name, **best_params_clean})

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.to_csv(OUTPUT_DIR / f'gridsearch_cv_results_{model_name}_bbb_relevant.csv', index=False)

    pred_df = pd.DataFrame({'index': X_test.index, 'y_true': y_test.values, 'y_pred': y_pred, 'y_score': y_score})
    pred_df.to_csv(OUTPUT_DIR / f'predictions_{model_name}_bbb_relevant.csv', index=False)
    prediction_frames.append(pred_df.assign(model=model_name))
    joblib.dump(best_model, OUTPUT_DIR / f'{model_name}_bbb_relevant_best_model.pkl')

    print(f"Best hyperparameters: {best_params_clean}")
    print(f"Best CV ROC-AUC: {search.best_score_:.4f} ± {cv_metrics['cv_roc_auc_std']:.4f}")
    print(f"Test Acc={metrics['accuracy']:.4f} | F1={metrics['f1']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | MCC={metrics['mcc']:.4f}")

official_results = pd.DataFrame(official_rows).sort_values('roc_auc', ascending=False)
hyperparameters_df = pd.DataFrame(hyperparameter_rows)
official_results.to_csv(OUTPUT_DIR / 'results_bbb_relevant_official.csv', index=False)
hyperparameters_df.to_csv(OUTPUT_DIR / 'best_hyperparameters_bbb_relevant.csv', index=False)
pd.concat(prediction_frames, ignore_index=True).to_csv(OUTPUT_DIR / 'predictions_all_models_bbb_relevant.csv', index=False)

display(official_results)
display(hyperparameters_df)



Tuning LogReg with GridSearchCV + 3-fold Stratified CV
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best hyperparameters: {'clf__C': 10, 'sampler': 'RandomUnderSampler', 'selector__k': 'all'}
Best CV ROC-AUC: 0.8464 ± 0.0027
Test Acc=0.7877 | F1=0.8403 | ROC-AUC=0.8590 | MCC=0.5382

Tuning KNN with GridSearchCV + 3-fold Stratified CV
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best hyperparameters: {'clf__n_neighbors': 15, 'clf__p': 2, 'clf__weights': 'distance', 'sampler': 'none', 'selector__k': 'all'}
Best CV ROC-AUC: 0.8556 ± 0.0038
Test Acc=0.8206 | F1=0.8766 | ROC-AUC=0.8718 | MCC=0.5508

Tuning RF with GridSearchCV + 3-fold Stratified CV
Fitting 3 folds for each of 32 candidates, totalling 96 fits
Best hyperparameters: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 3, 'clf__n_estimators': 800, 'sampler': 'none', 'selector__k': 'all'}
Best CV ROC-AUC: 0.8966 ± 0.0031
Test Acc=0.8451 | F1=0.8900 | ROC-AUC=0.9104 | MCC

KeyboardInterrupt: 

## Load Pre-Trained Models and Run Stratified Bootstrap Trials

In [6]:
def stratified_bootstrap_indices(y, seed):
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y)
    indices = []
    for cls in np.unique(y_array):
        cls_idx = np.where(y_array == cls)[0]
        indices.append(rng.choice(cls_idx, size=len(cls_idx), replace=True))
    out = np.concatenate(indices)
    rng.shuffle(out)
    return out


def display(obj=None, *args, **kwargs):
    try:
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)
    except Exception:
        print(repr(obj))


def savefig(fig, name, fig_dir=FIGURE_DIR):
    fig.savefig(fig_dir / f'{name}.png', bbox_inches='tight', dpi=600)
    fig.savefig(fig_dir / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig)


# Load all trained models
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

sns.set_theme(context='paper', style='whitegrid', font_scale=1.35)
plt.rcParams.update({'savefig.dpi': 600, 'pdf.fonttype': 42, 'ps.fonttype': 42})

trained_models = {}
model_names_all = ['LogReg', 'KNN', 'RF', 'ET']

for model_name in model_names_all:
    model_path = OUTPUT_DIR / f'{model_name}_bbb_relevant_best_model.pkl'
    if model_path.exists():
        trained_models[model_name] = joblib.load(model_path)
        print(f'Loaded: {model_name}')
    else:
        print(f'Not found: {model_path}')

# Generate official test set results
official_rows = []
prediction_frames = []

for model_name, model in trained_models.items():
    y_pred, y_score = predict_scores(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    official_rows.append({'model': model_name, 'split': 'bbb_relevant_8020_official', **metrics})
    prediction_frames.append(pd.DataFrame({
        'model': model_name, 'index': X_test.index, 
        'y_true': y_test.values, 'y_pred': y_pred, 'y_score': y_score
    }))

official = pd.DataFrame(official_rows).sort_values('roc_auc', ascending=False)
predictions = pd.concat(prediction_frames, ignore_index=True)

# Run stratified bootstrap trials
trial_rows = []
trial_prediction_frames = []

for trial_id, seed in enumerate(RANDOM_SEEDS, start=1):
    boot_idx = stratified_bootstrap_indices(y_test.values, seed)
    X_trial = X_test.iloc[boot_idx]
    y_trial = y_test.iloc[boot_idx]
    
    for model_name, model in trained_models.items():
        y_pred, y_score = predict_scores(model, X_trial)
        metrics = compute_metrics(y_trial, y_pred, y_score)
        trial_rows.append({'trial': trial_id, 'seed': seed, 'model': model_name, **metrics})
        trial_prediction_frames.append(pd.DataFrame({
            'trial': trial_id, 'seed': seed, 'model': model_name, 
            'index': X_trial.index, 'y_true': y_trial.values, 'y_pred': y_pred, 'y_score': y_score
        }))

trials = pd.DataFrame(trial_rows)
trial_predictions = pd.concat(trial_prediction_frames, ignore_index=True)

# Summary statistics
summary = trials.groupby('model').agg(['mean', 'std'])
summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
summary = summary.reset_index().sort_values('roc_auc_mean', ascending=False)

# Save results
trials.to_csv(OUTPUT_DIR / 'trials_results_3runs_bbb_relevant.csv', index=False)
trial_predictions.to_csv(OUTPUT_DIR / 'trial_predictions_3runs_bbb_relevant.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'trials_summary_mean_std_bbb_relevant.csv', index=False)
official.to_csv(OUTPUT_DIR / 'results_official_bbb_relevant.csv', index=False)

display(official)
display(summary)

Loaded: LogReg
Loaded: KNN
Loaded: RF
Loaded: ET
    model                       split  accuracy  balanced_accuracy  precision    recall  specificity        f1   roc_auc  average_precision       mcc
2      RF  bbb_relevant_8020_official  0.845070           0.816789   0.895678  0.884474     0.749104  0.890041  0.910381           0.967328  0.628082
3      ET  bbb_relevant_8020_official  0.846635           0.828455   0.908046  0.871965     0.784946  0.889640  0.909656           0.967281  0.640107
1     KNN  bbb_relevant_8020_official  0.820553           0.764111   0.855143  0.899191     0.629032  0.876614  0.871758           0.933158  0.550830
0  LogReg  bbb_relevant_8020_official  0.787689           0.787409   0.900000  0.788079     0.786738  0.840330  0.859033           0.935370  0.538166
    model  trial_mean  trial_std  seed_mean   seed_std  accuracy_mean  accuracy_std  balanced_accuracy_mean  balanced_accuracy_std  precision_mean  precision_std  recall_mean  recall_std  specificity_m

## Publication-Ready Performance Figures

In [7]:
order = summary.sort_values('roc_auc_mean', ascending=False)['model'].tolist()
metric_panels = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision', 'mcc']
plot_df = summary.set_index('model').loc[order]

# Combined metrics panel (9-subplot)
fig, axes = plt.subplots(3, 3, figsize=(18, 14), constrained_layout=True)
for ax, metric in zip(axes.ravel(), metric_panels):
    means = plot_df[f'{metric}_mean']
    stds = plot_df[f'{metric}_std'].fillna(0)
    ax.bar(order, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
    ax.set_title(metric.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Mean ± SD')
    ax.tick_params(axis='x', rotation=35)
    if metric != 'mcc':
        ax.set_ylim(0, 1.02)
fig.suptitle('BBB-Relevant Descriptors: Bootstrap Trial Mean ± SD', fontsize=18, fontweight='bold')
savefig(fig, 'bbb_relevant_combined_metrics_mean_std_panel')

# Spider/radar chart
radar_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]
fig = plt.figure(figsize=(10, 10))
ax = plt.subplot(111, polar=True)
for model in order:
    row = summary[summary['model'] == model].iloc[0]
    values = [row[f'{m}_mean'] for m in radar_metrics] + [row[f'{radar_metrics[0]}_mean']]
    ax.plot(angles, values, linewidth=2, label=model)
    ax.fill(angles, values, alpha=0.08)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', ' ').title() for m in radar_metrics], fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('BBB-Relevant Descriptors\nSpider Chart of Mean Metrics', fontsize=17, fontweight='bold', pad=28)
ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.12), frameon=True)
savefig(fig, 'bbb_relevant_spider_radar_metrics')

# ROC curves (official test set)
fig, ax = plt.subplots(figsize=(8, 7))
for model in order:
    g = predictions[predictions['model'] == model]
    fpr, tpr, _ = roc_curve(g['y_true'], g['y_score'])
    ax.plot(fpr, tpr, linewidth=2, label=f'{model} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
ax.set_title('BBB-Relevant Descriptors: ROC Curves (Official Test Set)', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', frameon=True)
savefig(fig, 'bbb_relevant_roc_curves')

# Per-model bar graphs
individual_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision']
for model in order:
    row = summary[summary['model'] == model].iloc[0]
    labels = [m.replace('_', ' ').title() for m in individual_metrics]
    means = [row[f'{m}_mean'] for m in individual_metrics]
    stds = [row[f'{m}_std'] for m in individual_metrics]

    fig, ax = plt.subplots(figsize=(10, 5.8))
    ax.bar(labels, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
    ax.set_title(f'{model}: Bootstrap Trial Mean Metrics ± SD', fontweight='bold')
    ax.set_ylabel('Mean ± SD')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=35)
    savefig(fig, f'bbb_relevant_{model}_individual_metric_profile')

print(f'\n✓ All figures saved to: {FIGURE_DIR}')
print(f'✓ Summary CSV: {OUTPUT_DIR / "trials_summary_mean_std_bbb_relevant.csv"}')


✓ All figures saved to: ../figures/new_models
✓ Summary CSV: ../output/models/bbb_relevant_padel_v2/trials_summary_mean_std_bbb_relevant.csv
